# TT-14 — ElasticNet: Dự báo tải sưởi (Y1) và tải làm mát (Y2)
### Bộ dữ liệu Energy Efficiency (UCI) — 768 dòng × 8 đặc trưng

**Mục tiêu:** Chứng minh ElasticNet xử lý tốt đa cộng tuyến (X1, X2, X4, X5 tương quan gần hoàn hảo)
tốt hơn Lasso, đồng thời vẫn loại được biến vô ích tốt hơn Ridge.

Notebook đi theo đúng 10 bước trong README. Mỗi cell nên chạy tuần tự, không skip.


## 0. Import thư viện & thiết lập

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
)
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)


## Bước 1 — Nạp dữ liệu, tính ma trận tương quan + VIF
Xác nhận đa cộng tuyến nặng giữa X1, X2, X4, X5 (đúng như README cảnh báo).


In [ ]:
df = pd.read_excel("/mnt/user-data/uploads/ENB2012_data.xlsx")

# Đổi tên cột cho dễ đọc (giữ nguyên X1..X8, Y1, Y2 để khớp README)
col_meaning = {
    "X1": "Relative_Compactness",
    "X2": "Surface_Area",
    "X3": "Wall_Area",
    "X4": "Roof_Area",
    "X5": "Overall_Height",
    "X6": "Orientation",          # phân loại
    "X7": "Glazing_Area",
    "X8": "Glazing_Area_Dist",    # phân loại
    "Y1": "Heating_Load",
    "Y2": "Cooling_Load",
}

print("Kích thước dữ liệu:", df.shape)
print("\nKiểu dữ liệu:")
print(df.dtypes)
print("\nGiá trị thiếu (nếu có):")
print(df.isnull().sum())

df.head()


In [ ]:
# Ma trận tương quan (chỉ biến số liên tục, X6/X8 sẽ xử lý one-hot ở Bước 2)
numeric_cols = ["X1", "X2", "X3", "X4", "X5", "X7"]
corr = df[numeric_cols + ["Y1", "Y2"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Ma trận tương quan giữa các biến")
plt.tight_layout()
plt.savefig("reports/correlation_heatmap.png", dpi=150)
plt.show()

print("\n⚠️ Kiểm tra: |r| > 0.95 giữa X1, X2, X4, X5?")
pairs_high = []
for i, a in enumerate(numeric_cols):
    for b in numeric_cols[i+1:]:
        r = df[a].corr(df[b])
        if abs(r) > 0.95:
            pairs_high.append((a, b, round(r, 3)))
print(pairs_high)


In [ ]:
# VIF (Variance Inflation Factor) — đo đa cộng tuyến trực tiếp
# VIF > 10 (nhiều tài liệu dùng ngưỡng 5 hoặc 10) => đa cộng tuyến nghiêm trọng
X_vif = df[numeric_cols].copy()
X_vif = (X_vif - X_vif.mean()) / X_vif.std()  # chuẩn hoá trước khi tính VIF cho ổn định số học
X_vif = X_vif.assign(const=1)  # statsmodels cần cột hằng số

vif_data = pd.DataFrame()
vif_data["feature"] = numeric_cols
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i) for i in range(len(numeric_cols))
]
vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)
vif_data.to_csv("reports/vif_table.csv", index=False)

print(vif_data)
print("\n👉 Biến nào VIF cao chứng tỏ đa cộng tuyến nặng, đúng như README cảnh báo về nhóm X1-X2-X4-X5.")


## Bước 2 — One-hot encode X6, X8; chuẩn hoá các biến số
X6 (hướng nhà) và X8 (phân bố kính) là biến **phân loại** dù mã hoá bằng số →
phải one-hot, tuyệt đối không để dạng số có thứ tự (kẻo model hiểu nhầm "hướng 4 > hướng 2").


In [ ]:
numeric_features = ["X1", "X2", "X3", "X4", "X5", "X7"]
categorical_features = ["X6", "X8"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features),
    ]
)

X = df[numeric_features + categorical_features]
y1 = df["Y1"]  # tải sưởi
y2 = df["Y2"]  # tải làm mát

print("Số cột sau one-hot (ước tính):", len(numeric_features) +
      (df["X6"].nunique() - 1) + (df["X8"].nunique() - 1))


## Bước 3 — Baseline: DummyRegressor + Linear Regression
Đây là mốc để so sánh: nếu ElasticNet không tốt hơn baseline rõ rệt thì có vấn đề.


In [ ]:
def evaluate_baseline(X, y, target_name, random_state=RANDOM_STATE):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    results = {}
    for name, model in [
        ("Dummy (mean)", DummyRegressor(strategy="mean")),
        ("Linear Regression", LinearRegression()),
    ]:
        pipe = Pipeline([("prep", preprocessor), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)
        results[name] = {"RMSE": rmse, "R2": r2}

    return pd.DataFrame(results).T, (X_train, X_test, y_train, y_test)

baseline_Y1, split_Y1 = evaluate_baseline(X, y1, "Y1")
baseline_Y2, split_Y2 = evaluate_baseline(X, y2, "Y2")

print("=== Baseline Y1 (Heating Load) ===")
print(baseline_Y1)
print("\n=== Baseline Y2 (Cooling Load) ===")
print(baseline_Y2)


## Bước 4 — Chạy 3 model trên CÙNG dữ liệu: Ridge · Lasso · ElasticNet (cho Y1 trước)
Dùng `*CV` để tự dò `alpha` bằng cross-validation 5-fold (đủ ổn định với 768 dòng, đúng khuyến cáo README).
`ElasticNetCV` còn dò thêm `l1_ratio` — chính là điểm khác biệt cốt lõi.


In [ ]:
def fit_regularized_models(X_train, y_train, l1_ratios=(0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0)):
    alphas = np.logspace(-4, 1, 100)

    models = {
        "Ridge": Pipeline([
            ("prep", preprocessor),
            ("model", RidgeCV(alphas=alphas, cv=5)),
        ]),
        "Lasso": Pipeline([
            ("prep", preprocessor),
            ("model", LassoCV(alphas=alphas, cv=5, max_iter=50000, random_state=RANDOM_STATE)),
        ]),
        "ElasticNet": Pipeline([
            ("prep", preprocessor),
            ("model", ElasticNetCV(
                l1_ratio=list(l1_ratios),
                alphas=alphas,
                cv=5, max_iter=50000, random_state=RANDOM_STATE)),
        ]),
    }

    for name, pipe in models.items():
        pipe.fit(X_train, y_train)

    return models

X_train_Y1, X_test_Y1, y_train_Y1, y_test_Y1 = split_Y1
models_Y1 = fit_regularized_models(X_train_Y1, y_train_Y1)

print("alpha tối ưu:")
print("  Ridge      :", models_Y1["Ridge"]["model"].alpha_)
print("  Lasso      :", models_Y1["Lasso"]["model"].alpha_)
print("  ElasticNet :", models_Y1["ElasticNet"]["model"].alpha_,
      "| l1_ratio:", models_Y1["ElasticNet"]["model"].l1_ratio_)


## Bước 5 — ⭐ Bảng so sánh: mỗi model giữ bao nhiêu biến? RMSE bao nhiêu?


In [ ]:
def get_feature_names(preprocessor):
    num_names = numeric_features
    cat_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features))
    return num_names + cat_names

def compare_models(models, X_test, y_test, target_name):
    rows = []
    for name, pipe in models.items():
        pred = pipe.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)
        coefs = pipe["model"].coef_
        n_nonzero = int(np.sum(np.abs(coefs) > 1e-6))
        rows.append({
            "Model": name,
            "RMSE": round(rmse, 4),
            "R2": round(r2, 4),
            "So_bien_giu_lai": n_nonzero,
            "Tong_so_bien": len(coefs),
        })
    result = pd.DataFrame(rows).set_index("Model")
    print(f"=== So sánh 3 model cho {target_name} ===")
    print(result)
    return result

feature_names = get_feature_names(models_Y1["ElasticNet"]["prep"])
compare_Y1 = compare_models(models_Y1, X_test_Y1, y_test_Y1, "Y1 (Heating Load)")
compare_Y1.to_csv("reports/so_sanh_3_model_Y1.csv")


In [ ]:
# Biểu đồ so sánh RMSE + số biến giữ lại
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

compare_Y1["RMSE"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("RMSE theo model (Y1)")
axes[0].set_ylabel("RMSE")

compare_Y1["So_bien_giu_lai"].plot(kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("Số biến giữ lại (Y1)")
axes[1].set_ylabel("Số biến (khác 0)")

plt.tight_layout()
plt.savefig("reports/so_sanh_3_model.png", dpi=150)
plt.show()


## Bước 6 — ⭐ Kiểm chứng HIỆU ỨNG GOM NHÓM
Nhóm biến dính chặt nhau: **X1 (độ gọn), X2 (diện tích bề mặt), X4 (diện tích mái), X5 (chiều cao)**.
- Lasso kỳ vọng: chọn 1 biến trong nhóm, hệ số các biến còn lại ≈ 0.
- ElasticNet kỳ vọng: giữ lại nhiều hơn 1 biến trong nhóm (grouping effect).


In [ ]:
group_vars = ["X1", "X2", "X4", "X5"]

coef_table = pd.DataFrame({
    name: pipe["model"].coef_ for name, pipe in models_Y1.items()
}, index=feature_names)

print("=== Hệ số hồi quy (đã chuẩn hoá) cho nhóm biến tương quan ===")
print(coef_table.loc[group_vars].round(4))

print("\n=== Toàn bộ hệ số (để tham khảo) ===")
print(coef_table.round(4))

print("\n👉 Nhận xét cần điền vào README phần 'HIỆU ỨNG GOM NHÓM':")
for var in group_vars:
    lasso_c = coef_table.loc[var, "Lasso"]
    en_c = coef_table.loc[var, "ElasticNet"]
    print(f"  {var}: Lasso={lasso_c:.4f}  |  ElasticNet={en_c:.4f}"
          f"  {'-> Lasso đã LOẠI biến này' if abs(lasso_c) < 1e-6 else ''}")


## Bước 7 — Vẽ heatmap RMSE theo lưới (alpha × l1_ratio)
Dùng GridSearchCV thủ công (không dùng ElasticNetCV) để lấy được điểm RMSE ứng với TỪNG cặp
(alpha, l1_ratio) — phục vụ trực quan hoá, khác với ElasticNetCV chỉ trả về điểm tối ưu.


In [ ]:
alpha_grid = np.logspace(-3, 1, 15)
l1_ratio_grid = np.array([0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0])

param_grid = {
    "model__alpha": alpha_grid,
    "model__l1_ratio": l1_ratio_grid,
}

en_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", ElasticNet(max_iter=50000, random_state=RANDOM_STATE)),
])

grid = GridSearchCV(
    en_pipe, param_grid, cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
grid.fit(X_train_Y1, y_train_Y1)

results_df = pd.DataFrame(grid.cv_results_)
rmse_pivot = results_df.pivot_table(
    index="param_model__l1_ratio",
    columns="param_model__alpha",
    values="mean_test_score",
) * -1  # chuyển neg RMSE -> RMSE dương

plt.figure(figsize=(11, 5))
sns.heatmap(rmse_pivot, cmap="viridis_r", annot=False,
            xticklabels=[f"{a:.3f}" for a in alpha_grid])
plt.title("CV RMSE theo lưới alpha × l1_ratio (Y1)")
plt.xlabel("alpha")
plt.ylabel("l1_ratio")
plt.tight_layout()
plt.savefig("reports/heatmap_alpha_l1ratio.png", dpi=150)
plt.show()

print("Điểm tốt nhất trên lưới:", grid.best_params_, "| RMSE:", -grid.best_score_)


## Bước 8 — Làm cả hai nhãn Y1 và Y2 → so sánh biến quan trọng


In [ ]:
X_train_Y2, X_test_Y2, y_train_Y2, y_test_Y2 = split_Y2
models_Y2 = fit_regularized_models(X_train_Y2, y_train_Y2)

print("alpha / l1_ratio tối ưu (Y2):")
print("  ElasticNet :", models_Y2["ElasticNet"]["model"].alpha_,
      "| l1_ratio:", models_Y2["ElasticNet"]["model"].l1_ratio_)

compare_Y2 = compare_models(models_Y2, X_test_Y2, y_test_Y2, "Y2 (Cooling Load)")
compare_Y2.to_csv("reports/so_sanh_3_model_Y2.csv")


In [ ]:
# So sánh hệ số ElasticNet giữa Y1 và Y2 -> biến nào quan trọng cho sưởi vs làm mát
coef_compare = pd.DataFrame({
    "ElasticNet_Y1": models_Y1["ElasticNet"]["model"].coef_,
    "ElasticNet_Y2": models_Y2["ElasticNet"]["model"].coef_,
}, index=feature_names)

coef_compare["Chenh_lech"] = (coef_compare["ElasticNet_Y1"] - coef_compare["ElasticNet_Y2"]).abs()
coef_compare = coef_compare.sort_values("Chenh_lech", ascending=False)

print(coef_compare.round(4))

coef_compare[["ElasticNet_Y1", "ElasticNet_Y2"]].plot(kind="bar", figsize=(10, 5))
plt.title("So sánh hệ số ElasticNet: Y1 (sưởi) vs Y2 (làm mát)")
plt.ylabel("Hệ số (đã chuẩn hoá)")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("reports/coef_compare_Y1_Y2.png", dpi=150)
plt.show()


## Bước 9 — Kiểm tra ổn định: bootstrap 100 lần
Hệ số ElasticNet dao động bao nhiêu qua 100 lần resample? Nếu dao động lớn → model kém ổn định.


In [ ]:
def bootstrap_coefficients(X, y, best_alpha, best_l1_ratio, n_boot=100, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = len(X)
    coef_list = []

    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        X_boot = X.iloc[idx]
        y_boot = y.iloc[idx]

        pipe = Pipeline([
            ("prep", preprocessor),
            ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio,
                                  max_iter=50000, random_state=random_state)),
        ])
        pipe.fit(X_boot, y_boot)
        coef_list.append(pipe["model"].coef_)

    return np.array(coef_list)

best_alpha_Y1 = models_Y1["ElasticNet"]["model"].alpha_
best_l1_Y1 = models_Y1["ElasticNet"]["model"].l1_ratio_

boot_coefs = bootstrap_coefficients(X, y1, best_alpha_Y1, best_l1_Y1, n_boot=100)

boot_summary = pd.DataFrame({
    "feature": feature_names,
    "mean_coef": boot_coefs.mean(axis=0),
    "std_coef": boot_coefs.std(axis=0),
    "cv_percent": np.abs(boot_coefs.std(axis=0) / (boot_coefs.mean(axis=0) + 1e-9)) * 100,
}).sort_values("std_coef", ascending=False)

print(boot_summary.round(4))
boot_summary.to_csv("reports/bootstrap_stability.csv", index=False)


In [ ]:
# Boxplot phân phối hệ số qua 100 lần bootstrap
plt.figure(figsize=(11, 5))
plt.boxplot(boot_coefs, labels=feature_names, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="red", linestyle="--", linewidth=0.8)
plt.title("Phân phối hệ số ElasticNet qua 100 lần bootstrap (Y1)")
plt.tight_layout()
plt.savefig("reports/bootstrap_boxplot.png", dpi=150)
plt.show()


## Lưu model đã huấn luyện

In [ ]:
joblib.dump(models_Y1["ElasticNet"], "models/elasticnet_Y1.joblib")
joblib.dump(models_Y2["ElasticNet"], "models/elasticnet_Y2.joblib")
print("Đã lưu models/elasticnet_Y1.joblib và models/elasticnet_Y2.joblib")


## Bước 10 — ✍️ Đề xuất thay đổi thiết kế

Chạy cell dưới để in ra các biến có ảnh hưởng lớn nhất (theo trị tuyệt đối hệ số ElasticNet
đã chuẩn hoá) cho từng nhãn — dùng kết quả này để viết 3 đề xuất thiết kế cụ thể trong README
(mục "HIỆU ỨNG GOM NHÓM").


In [ ]:
def top_features(pipe, feature_names, target_name, top_n=5):
    coefs = pipe["model"].coef_
    s = pd.Series(coefs, index=feature_names).sort_values(key=np.abs, ascending=False)
    print(f"--- Top {top_n} biến ảnh hưởng nhất tới {target_name} ---")
    print(s.head(top_n).round(4))
    print()
    return s

top_Y1 = top_features(models_Y1["ElasticNet"], feature_names, "Y1 (Heating Load)")
top_Y2 = top_features(models_Y2["ElasticNet"], feature_names, "Y2 (Cooling Load)")

print("""
GỢI Ý VIẾT ĐỀ XUẤT (điền dựa trên kết quả top features ở trên):
  1. Nếu Overall_Height / Roof_Area có hệ số dương lớn -> cân nhắc giảm chiều cao
     tầng hoặc tối ưu diện tích mái để giảm tải sưởi/làm mát.
  2. Nếu Glazing_Area (X7) có hệ số dương -> giảm diện tích kính hoặc dùng kính
     cách nhiệt để giảm tải làm mát vào mùa hè.
  3. Nếu Relative_Compactness (X1) có hệ số âm mạnh -> tăng độ gọn của toà nhà
     (giảm tỉ lệ diện tích bề mặt / thể tích) để giảm thất thoát nhiệt.
""")


## Tổng kết

Đối chiếu với tiêu chí hoàn thành trong README:
- [x] Bảng VIF chứng minh đa cộng tuyến (`reports/vif_table.csv`)
- [x] Bảng so sánh Ridge / Lasso / ElasticNet — số biến giữ + RMSE (`reports/so_sanh_3_model_Y1.csv`, `_Y2.csv`)
- [x] Hiệu ứng gom nhóm: xem cell Bước 6
- [x] Heatmap alpha × l1_ratio (`reports/heatmap_alpha_l1ratio.png`)
- [x] Làm đủ cả 2 nhãn Y1 và Y2, có so sánh (cell Bước 8)
- [x] Ý nghĩa l1_ratio máy chọn — xem giá trị `l1_ratio_` in ra ở Bước 4/8
- [ ] So sánh RMSE với baseline — đối chiếu bảng Bước 3 và Bước 5

**Mức tham chiếu README:** R² ~0.90–0.92 cho Y1. So kết quả `compare_Y1` ở trên với mức này.
